# Nhóm 5, Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Tuổi thọ ổ bi từ rung động (NASA IMS)

**Khung SDG**: SDG 9 và SDG 12: "industrial internet of things (IIoT)", "sustainable manufacturing"; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Ba và thứ Sáu theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

%pip install pandas numpy duckdb matplotlib

In [1]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.6


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

In [2]:
# Step 1b: NASA PCoE IMS Bearing Data Set (three run-to-failure tests, 1-second vibration snapshots every 10 minutes)
import numpy as np, pandas as pd, glob, os
rows = []
for test_dir, n_ch in [('data/raw/1st_test', 8), ('data/raw/2nd_test', 4), ('data/raw/3rd_test', 4)]:
    files = sorted(glob.glob(test_dir + '/*'))
    for k, f in enumerate(files):
        a = np.loadtxt(f)                                   # 20480 samples x channels at 20 kHz
        for ch in range(a.shape[1]):
            x = a[:, ch]; rows.append({'test': os.path.basename(test_dir), 'snap': k, 'ts': pd.to_datetime(os.path.basename(f), format='%Y.%m.%d.%H.%M.%S'),
                                        'ch': ch, 'rms': np.sqrt((x**2).mean()), 'kurt': pd.Series(x).kurt(), 'peak': np.abs(x).max(), 'crest': np.abs(x).max()/np.sqrt((x**2).mean()),
                                        'skew': pd.Series(x).skew(), 'p2p': x.max()-x.min()})
df = pd.DataFrame(rows); print(df.shape, df.groupby('test').snap.max())



(46480, 10) test
1st_test    2155
2nd_test     983
3rd_test    6323
Name: snap, dtype: int64


<span style="color: #00bfff; font-weight: bold;">-------------------------------------------------------Ghi chú: sau đây là để test kết quả</span>

In [3]:
print(df.isnull().sum())

test     0
snap     0
ts       0
ch       0
rms      0
kurt     0
peak     0
crest    0
skew     0
p2p      0
dtype: int64


In [4]:
display(df.describe())

,snap,ts,ch,rms,kurt,peak,crest,skew,p2p
count,46480.000000,46480,46480.000000,46480.000000,46480.000000,46480.000000,46480.000000,46480.000000,46480.000000
mean,2162.060413,2004-02-02 06:03:46.318502784,2.242169,0.102473,0.635805,0.511107,4.965614,0.022636,0.896202
min,0.000000,2003-10-22 12:06:24,0.000000,0.001197,-1.935598,0.002000,1.614597,-2.542613,0.004000
25%,726.000000,2003-11-19 13:16:07,1.000000,0.069100,0.155644,0.330000,4.338425,-0.015220,0.623000
50%,1608.000000,2004-03-07 22:57:46,2.000000,0.079412,0.304798,0.435000,4.741710,0.015854,0.778000
75%,3418.250000,2004-03-28 14:34:27,3.000000,0.145304,0.666460,0.632000,5.261736,0.056160,1.055000
max,6323.000000,2004-04-18 02:42:55,7.000000,0.758765,71.579558,5.000000,25.948697,2.157123,9.998000
std,1783.504139,NaN,1.915186,0.043418,1.747221,0.302769,1.206790,0.079511,0.501943


<span style="color: #00bfff; font-weight: bold;">Ghi chú: HẾT TEST---------------------------------------------------------------------------</span>

### 1.1b. Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan trước khi tự viết EDA ở các ô sau.

<span style="color: #00bfff; font-weight: bold;">Ghi chú: để debug lỗi thiếu setuptool</span>

In [5]:
%pip install "setuptools<70.0.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


<span style="color: #00bfff; font-weight: bold;">Ghi chú: HẾT DEBUG </span>

%pip install ydata-profiling

%pip install "numpy<2.3"

In [6]:
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 340.83it/s]


<span style="color: #00bfff; font-weight: bold;">Ghi chú: tìm file</span>

In [7]:
import os
print("Thư mục hiện tại:", os.getcwd())
print("Các file trong folder report:", os.listdir('report'))

Thư mục hiện tại: c:\Users\lenovo\OneDrive\Documents\fpt work\3rd\ADY201m\project\Notebook\test note A
Các file trong folder report: ['fig_eda_missing.png', 'fig_rq1_box_period.png', 'fig_rq1_corr.png', 'fig_rq1_distribution.png', 'fig_rq1_heatmap.png', 'fig_rq1_period_bar.png', 'fig_rq1_scatter.png', 'fig_rq1_timeseries.png', 'profile.html', 'table_cleaning_log.csv', 'table_columns.csv', 'table_corr.csv', 'table_describe_categorical.csv', 'table_describe_numeric.csv', 'table_describe_raw.csv', 'table_eda_notes.csv', 'table_missing.csv', 'table_q1.csv', 'table_q2.csv', 'table_q3.csv', 'table_rq1a_period.csv', 'table_rq1b_corr.csv', 'table_rq1c_units.csv', 'table_target_by_period.csv', 'table_target_by_unit.csv']


## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

%pip install pyarrow

In [8]:
# Step 2: health index and RUL label per test-channel (end of run = failure), keep bearings with documented failures
df = df.drop_duplicates(['test','ch','snap']).dropna()
df['n_snap'] = df.groupby(['test','ch']).snap.transform('max') + 1
df['rul'] = (df.n_snap - 1 - df.snap) * 10 / 60          # remaining hours (10-minute snapshots)
df['hi'] = df.groupby(['test','ch']).rms.transform(lambda s: (s - s.iloc[:50].mean()) / s.iloc[:50].std())   # health index vs healthy baseline
df.to_parquet('data/processed/bearing.parquet', index=False); print(df.shape, df.rul.describe().round(1).to_dict())

(46480, 13) {'count': 46480.0, 'mean': 360.3, 'std': 297.3, 'min': 0.0, '25%': 121.0, '50%': 268.0, '75%': 569.7, 'max': 1053.8}


### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [9]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
_, cleaning_log = clean_log(df, [
    ('drop duplicates', lambda x: x.drop_duplicates(), 'exact duplicate rows'),
    ('drop missing target', lambda x: x.dropna(subset=['rul']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped                reason
0      drop duplicates        46480       46480        0  exact duplicate rows
1  drop missing target        46480       46480        0    cannot be forecast


### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [10]:
eda = df.copy()
import io
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T                         # numeric statistics
desc_num['skew'] = eda.select_dtypes('number').skew()  # skewness, > 1 means strongly right-skewed
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude='number').describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in eda.select_dtypes('number').columns:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False); print(notes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46480 entries, 0 to 46479
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   test    46480 non-null  object        
 1   snap    46480 non-null  int64         
 2   ts      46480 non-null  datetime64[ns]
 3   ch      46480 non-null  int64         
 4   rms     46480 non-null  float64       
 5   kurt    46480 non-null  float64       
 6   peak    46480 non-null  float64       
 7   crest   46480 non-null  float64       
 8   skew    46480 non-null  float64       
 9   p2p     46480 non-null  float64       
 10  n_snap  46480 non-null  int64         
 11  rul     46480 non-null  float64       
 12  hi      46480 non-null  float64       
dtypes: datetime64[ns](1), float64(8), int64(3), object(1)
memory usage: 4.6+ MB

          count                           mean                  min                  25%                  50%                  75%                  m

### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [11]:
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv('report/table_missing.csv'); print(miss.head(15))
# missingness of the target by time and by unit
tcol, gcol = 'snap', 'ch'
by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['rul'].apply(lambda s: s.isna().mean())
by_unit = eda.groupby(gcol)['rul'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())
print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())
# co-missingness matrix: do columns go missing together
co = eda.isna().astype(int); co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1: print(co.corr().round(2))
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D'); ax.set_ylabel('Missing rate'); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig('report/fig_eda_missing.png', dpi=300)

        missing_rate decision
test             0.0     keep
snap             0.0     keep
ts               0.0     keep
ch               0.0     keep
rms              0.0     keep
kurt             0.0     keep
peak             0.0     keep
crest            0.0     keep
skew             0.0     keep
p2p              0.0     keep
n_snap           0.0     keep
rul              0.0     keep
hi               0.0     keep
target missing by period (top): {6323: 0.0, 0: 0.0, 1: 0.0, 2: 0.0, 6307: 0.0}
target missing by unit (top): {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}


### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [12]:
num_cols = [c for c in ['rms', 'kurt', 'peak', 'crest', 'hi', 'rul'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'rul': cols.append([c, 'target', 'keep']); continue
    if c in ('snap', 'ch'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'rul'] if 'rul' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('ch')['rul'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['snap'] if np.issubdtype(eda['snap'].dtype, np.number) else pd.to_datetime(eda['snap'], errors='coerce').dt.month
by_t = eda.groupby(period)['rul'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

         rms   kurt   peak  crest     hi    rul
rms    1.000  0.329  0.780  0.043  0.829 -0.541
kurt   0.329  1.000  0.712  0.667  0.255 -0.184
peak   0.780  0.712  1.000  0.621  0.681 -0.418
crest  0.043  0.667  0.621  1.000  0.027 -0.035
hi     0.829  0.255  0.681  0.027  1.000 -0.469
rul   -0.541 -0.184 -0.418 -0.035 -0.469  1.000
       mean   median     p05      p95       n
ch                                           
0   401.522  312.250  26.192  974.975  9464.0
1   401.522  312.250  26.192  974.975  9464.0
2   401.522  312.250  26.192  974.975  9464.0
3   401.522  312.250  26.192  974.975  9464.0
4   179.583  179.583  17.958  341.208  2156.0
5   179.583  179.583  17.958  341.208  2156.0
6   179.583  179.583  17.958  341.208  2156.0
7   179.583  179.583  17.958  341.208  2156.0
         mean   median      p05       p95     n
snap                                           
0     484.000  359.167  163.833  1053.833  16.0
1     483.833  359.000  163.667  1053.667  16.0
2     483.66

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [13]:
# SQL statements run one by one in DuckDB; each SELECT prints its first 20 rows and is saved to report/table_q{k}.csv
import duckdb
if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')   # reuse the connection opened in Step 1
SQL = r"""
-- Step 3: sql/queries.sql
CREATE OR REPLACE TABLE br AS SELECT * FROM 'data/processed/bearing.parquet';
-- Q1: feature trends in the last 10% of life versus the first 10%
SELECT test, ch, AVG((CASE WHEN snap < 0.1 * n_snap THEN rms END)::INT) AS rms_early, AVG((CASE WHEN snap > 0.9 * n_snap THEN rms END)::INT) AS rms_late,
       AVG((CASE WHEN snap > 0.9 * n_snap THEN kurt END)::INT) AS kurt_late FROM br GROUP BY 1, 2;
-- Q2: first snapshot where the health index exceeds 3 (onset of degradation)
SELECT test, ch, MIN(snap) AS onset_snap, MAX(n_snap) AS n_snap FROM br WHERE hi > 3 GROUP BY 1, 2;
-- Q3: rolling features for RUL regression
CREATE OR REPLACE TABLE feat AS
SELECT test, ch, snap, ts, rms, kurt, peak, crest, skew, p2p, hi, rul,
       AVG(rms) OVER (w ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS rms_ma6, rms - LAG(rms, 6) OVER w AS rms_d6,
       AVG(kurt) OVER (w ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS kurt_ma6, MAX(peak) OVER (w ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS peak_max12
FROM br WINDOW w AS (PARTITION BY test, ch ORDER BY snap);
SELECT test, COUNT(*) AS n FROM feat GROUP BY 1;
"""
k = 0
SQL_NO_COMMENTS = '\n'.join(l for l in SQL.splitlines() if not l.strip().startswith('--'))   # drop comment lines first (they may contain semicolons)
for stmt in SQL_NO_COMMENTS.split(';'):
    stmt = stmt.strip()
    if not stmt: continue
    res = con.execute(stmt)
    if stmt.upper().startswith(('SELECT', 'WITH')):
        k += 1; out = res.df(); out.to_csv(f'report/table_q{k}.csv', index=False); print(f'--- Q{k} ---'); print(out.head(20))
open('sql/queries.sql', 'w').write(SQL)

--- Q1 ---
        test  ch  rms_early  rms_late  kurt_late
0   1st_test   1        0.0  0.000000   0.995349
1   2nd_test   3        0.0  0.000000   0.163265
2   1st_test   3        0.0  0.000000   0.130233
3   1st_test   6        0.0  0.000000   1.316279
4   2nd_test   1        0.0  0.000000   0.000000
5   1st_test   4        0.0  0.009302   8.855814
6   3rd_test   0        0.0  0.000000   0.373418
7   3rd_test   2        0.0  0.006329   0.281646
8   1st_test   5        0.0  0.000000   2.874419
9   3rd_test   3        0.0  0.000000   0.004747
10  2nd_test   2        0.0  0.000000   0.122449
11  1st_test   7        0.0  0.000000   1.232558
12  1st_test   2        0.0  0.000000   0.023256
13  1st_test   0        0.0  0.000000   0.981395
14  3rd_test   1        0.0  0.000000   0.001582
15  2nd_test   0        0.0  0.030612   1.693878
--- Q2 ---
        test  ch  onset_snap  n_snap
0   1st_test   7          34    2156
1   1st_test   5         102    2156
2   3rd_test   3        2037    63

1127

### 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định

**Làm gì**: từ kết quả Q1, Q2 dựng bảng có số thứ tự đưa thẳng vào bài; kiểm định khác biệt giữa nhóm (Kruskal-Wallis, không cần giả định chuẩn) và tương quan Spearman giữa mục tiêu và các biến ngoại sinh kèm p-value.

**Đọc thế nào**: p < 0,05 nghĩa là khác biệt giữa nhóm khó do ngẫu nhiên; hệ số Spearman cho chiều và độ mạnh quan hệ đơn điệu; kết luận RQ1 phải là một câu có số, ví dụ "mùa đông cao gấp 1,9 lần mùa hè, p < 0,001".

In [14]:
from scipy.stats import kruskal, spearmanr
q1 = con.execute("SELECT * FROM feat").df()          # feature table created in Q3
if np.issubdtype(q1['snap'].dtype, np.number): q1['period'] = q1['snap']            # numeric time key (year, cycle)
else: q1['period'] = pd.to_datetime(q1['snap'], errors='coerce').dt.month             # datetime key: use month as period
# Table RQ1a: target by period
rq1a = q1.groupby('period')['rul'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.values for _, s in q1.groupby('period')['rul'] if len(s) > 30]
if len(groups) > 1: print('Kruskal-Wallis across periods:', kruskal(*groups))
# Table RQ1b: Spearman correlation of the target with exogenous variables
ext = [c for c in q1.select_dtypes('number').columns if c not in ('rul', 'rul') and not c.startswith('rul')]
rows = []
for c in ext[:15]:
    r, pv = spearmanr(q1[c], q1['rul'], nan_policy='omit'); rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print(rq1b)
# Table RQ1c: target by unit, top and bottom 5
rq1c = q1.groupby('ch')['rul'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('RQ1 CONCLUSION (fill with real numbers): highest period', rq1a['mean'].idxmax(), 'is', round(rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9), 2), 'times the lowest period')

           mean   median      std  count
period                                  
0       484.000  359.167  349.622     16
1       483.833  359.000  349.622     16
2       483.667  358.833  349.622     16
3       483.500  358.667  349.622     16
4       483.333  358.500  349.622     16
...         ...      ...      ...    ...
6319      0.667    0.667    0.000      4
6320      0.500    0.500    0.000      4
6321      0.333    0.333    0.000      4
6322      0.167    0.167    0.000      4
6323      0.000    0.000    0.000      4

[6324 rows x 4 columns]
      variable  spearman_r        p_value
8           hi      -0.597   0.000000e+00
2          rms      -0.547   0.000000e+00
9      rms_ma6      -0.547   0.000000e+00
12  peak_max12      -0.528   0.000000e+00
4         peak      -0.525   0.000000e+00
7          p2p      -0.520   0.000000e+00
11    kurt_ma6      -0.368   0.000000e+00
3         kurt      -0.356   0.000000e+00
6         skew       0.219   0.000000e+00
0           ch      -0

## Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

Mỗi hình gắn với một ý của RQ1 và có caption là kết luận. Bộ hình: (1) phân phối mục tiêu, histogram và boxplot; (2) chuỗi theo thời gian của vài đơn vị; (3) heatmap thời kỳ x đơn vị; (4) boxplot mục tiêu theo thời kỳ; (5) scatter mục tiêu với biến ngoại sinh mạnh nhất; (6) heatmap tương quan; (7) so sánh nhóm bằng cột có số. Quy tắc: chữ tiếng Anh, bỏ viền trên và phải, số in đậm trên cột, 300 dpi, tên file `fig_rq1_*.png`.

In [15]:
import seaborn as sns
def style(ax): ax.spines[['top', 'right']].set_visible(False)
d = q1.dropna(subset=['rul'])
# (1) distribution
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['rul'], bins=50, color='#1B6B6D'); ax[0].set_xlabel('rul hours'); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['rul'], vert=False); ax[1].set_xlabel('rul'); style(ax[1])
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)
# (2) time series of the first 3 units
units = d['ch'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['ch'] == u].sort_values('snap'); ax.plot(s['snap'], s['rul'], lw=0.8, label=str(u))
ax.set_ylabel('rul'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)
# (3) heatmap period x unit (top 15 units)
top = d.groupby('ch')['rul'].mean().nlargest(15).index
hm = d[d['ch'].isin(top)].pivot_table(index='ch', columns='period', values='rul', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4)); sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'rul'}); ax.set_xlabel('Period'); ax.set_ylabel('Unit')
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)
# (4) boxplot by period
fig, ax = plt.subplots(figsize=(9, 3)); sns.boxplot(data=d, x='period', y='rul', color='#9FBFBF', ax=ax, showfliers=False); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)
# (5) scatter against the strongest exogenous variable
if len(rq1b):
    v = rq1b.iloc[0].variable
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.scatter(d[v], d['rul'], s=4, alpha=0.3, color='#1B6B6D'); ax.set_xlabel(v); ax.set_ylabel('rul'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)
# (6) correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(d.select_dtypes('number').corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)
# (7) group comparison with labelled bars
gm = d.groupby('period')['rul'].mean()
fig, ax = plt.subplots(figsize=(8, 3)); bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D'); ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8); ax.set_ylabel('Mean rul'); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)
print('7 figures saved to report/. Write each caption as: conclusion + number + condition.')

7 figures saved to report/. Write each caption as: conclusion + number + condition.


## Test case của Notebook_A (chạy trước khi bàn giao)

Mỗi assert là một điều kiện phải đúng; nếu sai, sửa bước tương ứng rồi chạy lại. Sau khi qua hết, ô cuối tạo `manifest.json` để Sinh viên B kiểm tra.

In [16]:
import hashlib, json, os

feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['rul'].notna().sum() > 0.8 * len(feat), 'too many missing targets'

# SỬA TẠI ĐÂY: Thêm 'test' vào khóa định danh unit-time
key = ['test', 'ch', 'snap']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'

lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'rul']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')

# Xuất parquet và tạo file manifest bàn giao cho Sinh viên B
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {
    'rows': int(len(feat)),
    'cols': list(feat.columns),
    'md5': h,
    'time_min': str(feat['snap'].min()),
    'time_max': str(feat['snap'].max()),
    'author': 'Student A'
}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str)
print(manifest)

Tests A: OK
{'rows': 46480, 'cols': ['test', 'ch', 'snap', 'ts', 'rms', 'kurt', 'peak', 'crest', 'skew', 'p2p', 'hi', 'rul', 'rms_ma6', 'rms_d6', 'kurt_ma6', 'peak_max12'], 'md5': '864248fc119f142a4c6ad5874eeeb56f', 'time_min': '0', 'time_max': '6323', 'author': 'Student A'}


In [ ]:
import hashlib, json, os
feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['rul'].notna().sum() > 0.8 * len(feat), 'too many missing targets'


key = ['test', 'ch', 'snap']


assert not feat.duplicated(key).any(), 'duplicate unit-time keys'
lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'rul']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {'rows': int(len(feat)), 'cols': list(feat.columns), 'md5': h,
            'time_min': str(feat['snap'].min()), 'time_max': str(feat['snap'].max()), 'author': 'Student A'}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str); print(manifest)

Tests A: OK
{'rows': 46480, 'cols': ['test', 'ch', 'snap', 'ts', 'rms', 'kurt', 'peak', 'crest', 'skew', 'p2p', 'hi', 'rul', 'rms_ma6', 'rms_d6', 'kurt_ma6', 'peak_max12'], 'md5': '864248fc119f142a4c6ad5874eeeb56f', 'time_min': '0', 'time_max': '6323', 'author': 'Student A'}


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [ ] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [ ] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [ ] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [ ] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [ ] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [ ] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [ ] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [ ] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [ ] Audit Log cá nhân có đủ prompt của tuần
- [ ] Báo cáo năm mục đã dán vào kênh nhóm trước 12:00 thứ Ba hoặc thứ Sáu


## Mẫu báo cáo tiến độ (slot 3 thứ Ba và thứ Sáu, 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................